<!--
---
PURPOSE: Scaffold pose project and export frame samples.
REQUIRES:
  - outputs/video/session_{id}_frame_times.parquet
PRODUCES:
  - pose_projects/session_{id}/{sleap|dlc}/...
  - outputs/pose/session_{id}_frame_samples/frame_samples.csv
WHAT_NEXT: notebooks/07_Pose_to_Motifs_Feature_Engineering.ipynb
---
-->

# 06 Pose Estimation Setup (SLEAP or DLC)

**Purpose**
Scaffold pose project and export frame samples.

**Requires**
- `outputs/video/session_{id}_frame_times.parquet`

**Produces**
- `pose_projects/session_{id}/{sleap|dlc}/...`
- `outputs/pose/session_{id}_frame_samples/frame_samples.csv`

**What to run next**
- `notebooks/07_Pose_to_Motifs_Feature_Engineering.ipynb`

We scaffold a pose project and export frame samples with timestamps.


## Environment Setup
We add the repo `src/` to the Python path so notebooks can import shared modules.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

## Prerequisite Check
We parse the notebook header and validate required artifacts before running downstream steps.

In [ ]:
from pathlib import Path
from reports import parse_notebook_header, validate_prerequisites

nb_path = Path.cwd() / "notebooks" / "06_Pose_Estimation_Setup_SLEAP_or_DLC.ipynb"
header = parse_notebook_header(nb_path)
missing = validate_prerequisites(header.get("REQUIRES", []))
if missing:
    print("Missing prerequisites:")
    for item in missing:
        print(" -", item)
else:
    print("All prerequisites satisfied.")

## Step 1: Scaffold pose project
This creates the directory structure for your chosen tool.

In [ ]:
import json
from pathlib import Path
from io_sessions import load_sessions_csv, get_session_bundle
from config import get_config
from features_pose import sample_frame_indices, scaffold_pose_project, export_frame_samples

cfg = get_config()
sessions = load_sessions_csv()
SESSION_IDS = sessions.session_id.tolist()[:1]

for session_id in SESSION_IDS:
    bundle = get_session_bundle(session_id, sessions)
    manifest = bundle.load_video_manifest(prefer_download=False)
    stream = manifest.get("streams", [{}])[0]
    video_path = stream.get("file_path")
    frame_times = bundle.load_frame_times()
    if video_path and frame_times is not None:
        project_dir = scaffold_pose_project(session_id, cfg.pose_tool, cfg.pose_projects_dir)
        samples_dir = cfg.outputs_dir / "pose" / f"session_{session_id}_frame_samples"
        frame_idx = sample_frame_indices(frame_times, n_samples=50)
        export_frame_samples(Path(video_path), frame_idx, samples_dir, frame_times)
        print("Pose project:", project_dir)